In [1]:
print("all ok")

all ok


In [21]:
# load the model
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o")
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x00000211B25E3130>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000211B71BC430>, root_client=<openai.OpenAI object at 0x00000211B25E1450>, root_async_client=<openai.AsyncOpenAI object at 0x00000211B25E0F70>, model_name='gpt-4o', model_kwargs={}, openai_api_key=SecretStr('**********'))

In [1]:
from langchain.document_loaders import PyPDFDirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [5]:
pdf_docs=PyPDFDirectoryLoader("data/").load()

In [24]:
final_pdf

[Document(metadata={'producer': 'LuaHBTeX, Version 1.18.0 (TeX Live 2024)', 'creator': 'LaTeX with acmart 2024/12/28 v2.12 Typesetting articles for the Association for Computing Machinery and hyperref 2024-01-20 v7.01h Hypertext links for LaTeX', 'creationdate': '2025-06-11T18:43:26+00:00', 'moddate': '2025-06-11T18:43:26+00:00', 'ptex.fullbanner': 'This is LuaHBTeX, Version 1.18.0 (TeX Live 2024)', 'subject': '-  Networks  ->  Network performance evaluation.-  Security and privacy  ->  Information flow control.-  Computing methodologies  ->  Artificial intelligence.', 'title': 'AI5GTest: AI-Driven Specification-Aware Automated Testing and Validation of 5G O-RAN Components', 'trapped': '/False', 'source': 'data\\2506.10111v1.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}, page_content='AI5GTest: AI-Driven Specification-Aware Automated Testing and\nValidation of 5G O-RAN Components\nAbiodun Ganiyu∗\nNextG Wireless Lab\nNorth Carolina State University\nRaleigh, USA\naganiyu@ncsu.

In [7]:
# split the file
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=150,
    length_function=len
)
final_pdf=text_splitter.split_documents(pdf_docs)

In [9]:
# load the embedding model
from langchain.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
embedding_model=OpenAIEmbeddings(model="text-embedding-3-large")

In [10]:
# create the vector store
vector_store=FAISS.from_documents(final_pdf,embedding_model)

In [13]:
# create the retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 10})

In [14]:
prompt_template = """
        Answer the question based on the context provided below. 
        If the context does not contain sufficient information, respond with: 
        "I do not have enough information about this."

        Context: {context}

        Question: {question}

        Answer:"""


In [15]:
from langchain.prompts import PromptTemplate
prompt=PromptTemplate(
    template=prompt_template,
    input_variables=["context","question"]
)
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\n        Answer the question based on the context provided below. \n        If the context does not contain sufficient information, respond with: \n        "I do not have enough information about this."\n\n        Context: {context}\n\n        Question: {question}\n\n        Answer:')

In [17]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [19]:
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

In [22]:
# create the RAG chain
rag_chain=(
    {"context":retriever | format_docs, "question":RunnablePassthrough()}
    |prompt|llm|parser
)

In [25]:
response=rag_chain.invoke("tell  me about procedural flows for test cases based on 3GPP?")
print(response)

The procedural flows for test cases based on 3GPP are generated by a module called Gen-LLM. Gen-LLM dynamically generates the expected procedural flow by referencing relevant 3GPP (and O-RAN) specifications. These flows outline the signaling exchanges and protocol behaviors expected for the test case. The framework includes a human-in-the-loop mechanism, enabling testers to review and approve the AI-generated flows. Once approved, these procedural flows are validated by another module called Val-LLM, which checks them against observed signaling logs to ensure compliance and detect any deviations. If there are issues, a further module, Debug-LLM, performs root cause analysis.
